# Colab Worker D: Track D - Baselines & Final Merge 2 Evaluation
This notebook runs on **Colab Account 4** (or any free GPU instance).
- Runs external state-of-the-art reference watermark **TrustMark**
- Runs **Merge 2 Final Evaluation** on the 500 held-out test images against the never-seen **Stable Diffusion VAE** (`stabilityai/sd-vae-ft-mse`)
- Evaluates:
  1. Classical Spread-Spectrum (best band from Track B)
  2. Fixed Low-band Control
  3. Fixed Mid-band Control
  4. Fixed High-band Control
  5. Learned Encoder Stage 1 (without surrogate)
  6. Learned Encoder Stage 2 (with fitted surrogate)
  7. TrustMark Baseline
- Outputs `results/merge2_final_eval.csv` for Fig 4 & Table 1.

In [ ]:
# 1. GPU Check & Drive Mount
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
SHARED_DIR = '/content/drive/MyDrive/watermark_internship'
!mkdir -p $SHARED_DIR/results

In [ ]:
# 2. Clone repo & setup dependencies
!git clone https://github.com/ashhyyyy-vis/Internship.git /content/repo
%cd /content/repo
!pip install -q diffusers transformers accelerate compressai kornia lpips pandas scipy tqdm trustmark

In [ ]:
# 3. Unpack image splits from Google Drive
!mkdir -p /content/repo/data
!if [ -f $SHARED_DIR/processed_splits.zip ]; then cp $SHARED_DIR/processed_splits.zip /content/repo/data/; fi
!unzip -q /content/repo/data/processed_splits.zip -d /content/repo/data/

In [ ]:
# 4. Copy checkpoints from Drive
!cp -r $SHARED_DIR/checkpoints /content/repo/

In [ ]:
# 5. Run Unified Merge 2 Benchmark on Held-Out Test Set
!python -m track_d.harness \
    --manifest data/manifests/test_manifest.json \
    --images_dir data/processed/test \
    --out results/merge2_final_eval.csv \
    --attacks sd_vae compressai jpeg gaussian_noise gaussian_blur

!cp results/merge2_final_eval.csv $SHARED_DIR/results/
print('✓ Merge 2 Final Benchmark complete! Results saved to Drive.')